# M06 Tracing the Thoughts of a Large Language Model


## Build a toy trace, then compare it to a shared graph

Construct a tiny computation path locally, inspect which edges carry the answer, and then compare that pattern to the shared teaching artifact.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import math
import matplotlib.pyplot as plt
import numpy as np

tokens = ["Q", "3", "+", "4", "A"]
embeddings = np.array([
    [0.1, 0.0, 0.2],
    [1.2, 0.1, 0.0],
    [0.0, 0.2, 0.1],
    [0.9, 0.0, 0.2],
    [0.7, 0.4, 1.0],
])
Wq = np.array([[0.2, 0.7], [0.9, 0.1], [0.5, 0.6]])
Wk = np.array([[0.6, 0.3], [0.7, 0.2], [0.4, 0.8]])
Q = embeddings @ Wq
K = embeddings @ Wk
scores = Q[-1] @ K.T / math.sqrt(K.shape[-1])
weights = np.exp(scores - scores.max())
weights = weights / weights.sum()

number_values = np.array([0.0, 3.0, 0.0, 4.0, 0.0])
readout_weights = np.array([0.0, 1.0, 0.0, 1.0, 0.0])
token_contrib = weights * number_values * readout_weights
feature_scores = {
    "retrieve_left": token_contrib[1],
    "retrieve_right": token_contrib[3],
}
compose_score = sum(feature_scores.values())
toy_edges = [
    ("3", "retrieve_left", feature_scores["retrieve_left"]),
    ("4", "retrieve_right", feature_scores["retrieve_right"]),
    ("retrieve_left", "compose_sum", feature_scores["retrieve_left"]),
    ("retrieve_right", "compose_sum", feature_scores["retrieve_right"]),
    ("compose_sum", "answer=7", compose_score),
]

positions = {
    "3": (0.15, 0.75),
    "4": (0.15, 0.25),
    "retrieve_left": (0.45, 0.75),
    "retrieve_right": (0.45, 0.25),
    "compose_sum": (0.72, 0.5),
    "answer=7": (0.9, 0.5),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for source, target, score in toy_edges:
    x1, y1 = positions[source]
    x2, y2 = positions[target]
    axes[0].plot([x1, x2], [y1, y2], linewidth=2 + 6 * score, color="#c96a28", alpha=0.7)
for node, (x, y) in positions.items():
    color = "#123b63" if "retrieve" in node or "compose" in node else "#b5893a"
    axes[0].scatter(x, y, s=700, color=color)
    axes[0].text(x, y, node, ha="center", va="center", color="white", fontsize=8)
axes[0].set_title("Toy computation path")
axes[0].axis("off")

artifact = json.loads((root / "artifacts" / "m06_attribution_graph.json").read_text())
case = artifact["cases"][0]
artifact_scores = sorted((edge["score"] for edge in case["edges"]), reverse=True)
axes[1].bar(["left", "right", "sum"], [feature_scores["retrieve_left"], feature_scores["retrieve_right"], compose_score], color="#1f5f8b")
axes[1].plot(range(len(artifact_scores)), artifact_scores, marker="o", color="#c96a28")
axes[1].set_title("Toy scores vs artifact edge ordering")
axes[1].set_xlabel("ordered artifact edges")
plt.tight_layout()

print("Answer-position attention:", dict(zip(tokens, np.round(weights, 3))))
print("Toy path contributions:")
for source, target, score in toy_edges:
    print(f"  {source} -> {target}: {score:.3f}")

ablated_left = compose_score - feature_scores["retrieve_left"]
ablated_right = compose_score - feature_scores["retrieve_right"]
print("Compose score after ablating left retrieval:", round(float(ablated_left), 3))
print("Compose score after ablating right retrieval:", round(float(ablated_right), 3))


## Takeaway

Tracing becomes legible when you can move between a locally computed path and a larger shared attribution graph.


## Turn this notebook into research output

Research worksheet means this notebook is not complete when the cells finish. Use the templates in /templates and leave behind written outputs.


### Before you run

- Choose one path or subgraph you plan to explain in detail.
- Write down what a faithful slice of computation means in this context.
- Open the memo template and reserve a section called 'what the graph does not prove'.


### After you run

- Explain three high-contribution edges in plain language.
- Mark one ambiguity that would require a follow-up experiment.


### Ship these artifacts

- One graph walkthrough.
- One ambiguity note.
- One next experiment that would reduce uncertainty about the graph.


## Self-check questions

- Without simply reading back the labels, explain what the three most important contribution paths in this attribution graph are doing.
- What conclusion does the graph support, and what conclusion does it clearly not support?
- If you could run only one follow-up experiment to reduce the ambiguity in this graph, what would you choose?
- If you cannot answer at least two of these without rereading the note, revisit the article and your written outputs.
